<a href="https://colab.research.google.com/github/Raduana-Khawla/-ranga-store-fix/blob/main/LSTM(Hyperparameter).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load Dataset
CSV_FILE_PATH = "/content/DailyDelhiClimateTest.csv"

try:
    data = pd.read_csv(CSV_FILE_PATH)
    print(f"Dataset loaded successfully with shape: {data.shape}")
except FileNotFoundError as e:
    print(f"File not found. Ensure the file exists at {CSV_FILE_PATH}")
    raise e

# Display dataset columns
print("Dataset Columns:", data.columns)

# Step 2: Preprocess Dataset
# Selecting 'meantemp' as the target variable
target_column = "meantemp"

if target_column not in data.columns:
    raise ValueError(f"Target column '{target_column}' not found in the dataset.")

features = data.drop(['date', target_column], axis=1)  # Drop date and target column
target = data[target_column]

# Normalize features
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# Prepare sequences for LSTM
sequence_length = 30
X, y = [], []

for i in range(sequence_length, len(features_scaled)):
    X.append(features_scaled[i-sequence_length:i])
    y.append(target.values[i])  # Use values to avoid pandas Series issues

X = np.array(X)
y = np.array(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Define Model Builder
def build_and_evaluate_model(config, X_train, y_train, X_test, y_test):
    # Build LSTM model
    model = Sequential()
    for _ in range(config['Number of Layers']):
        model.add(LSTM(config['Number of Neurons'], return_sequences=False if _ == config['Number of Layers'] - 1 else True, input_shape=X_train.shape[1:]))
        model.add(Dropout(config['Dropout Rate']))
    model.add(Dense(1, activation=config['Activation Function']))

    optimizer = None
    if config['Optimizer'] == 'adam':
        optimizer = Adam()
    elif config['Optimizer'] == 'rmsprop':
        optimizer = RMSprop()
    elif config['Optimizer'] == 'sgd':
        optimizer = SGD()

    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    # Early stopping
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # Train model
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=config['Epochs'],
        batch_size=32,
        verbose=0,
        callbacks=[early_stopping]
    )

    # Evaluate on test set
    test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
    return {
        'Configuration': config,
        'Test MAE': test_mae,
        'Test Loss': test_loss
    }

# Step 4: Define Hyperparameter Options
activation_functions = ['relu', 'tanh']
neurons_list = [32, 64, 128]
dropout_rates = [0.2, 0.3, 0.4, 0.5]
layers_list = [1, 2, 3]
optimizers = ['adam', 'rmsprop', 'sgd']
epochs_list = [20, 50, 100]

# Step 5: Generate Random Configurations and Evaluate
results = []
for _ in range(200):  # Generate 200 random configurations
    config = {
        'Activation Function': random.choice(activation_functions),
        'Number of Neurons': random.choice(neurons_list),
        'Dropout Rate': random.choice(dropout_rates),
        'Number of Layers': random.choice(layers_list),
        'Optimizer': random.choice(optimizers),
        'Epochs': random.choice(epochs_list)
    }
    print(f"Evaluating configuration: {config}")
    result = build_and_evaluate_model(config, X_train, y_train, X_test, y_test)
    results.append(result)

# Step 6: Display Results
best_result = min(results, key=lambda x: x['Test MAE'])
print("\nBest Configuration:")
print(best_result)


Dataset loaded successfully with shape: (114, 5)
Dataset Columns: Index(['date', 'meantemp', 'humidity', 'wind_speed', 'meanpressure'], dtype='object')
Evaluating configuration: {'Activation Function': 'tanh', 'Number of Neurons': 32, 'Dropout Rate': 0.2, 'Number of Layers': 3, 'Optimizer': 'rmsprop', 'Epochs': 50}


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Evaluating configuration: {'Activation Function': 'relu', 'Number of Neurons': 32, 'Dropout Rate': 0.4, 'Number of Layers': 2, 'Optimizer': 'sgd', 'Epochs': 50}
Evaluating configuration: {'Activation Function': 'tanh', 'Number of Neurons': 32, 'Dropout Rate': 0.2, 'Number of Layers': 2, 'Optimizer': 'adam', 'Epochs': 50}
Evaluating configuration: {'Activation Function': 'tanh', 'Number of Neurons': 32, 'Dropout Rate': 0.4, 'Number of Layers': 3, 'Optimizer': 'rmsprop', 'Epochs': 20}
Evaluating configuration: {'Activation Function': 'relu', 'Number of Neurons': 128, 'Dropout Rate': 0.3, 'Number of Layers': 1, 'Optimizer': 'adam', 'Epochs': 50}
Evaluating configuration: {'Activation Function': 'tanh', 'Number of Neurons': 128, 'Dropout Rate': 0.4, 'Number of Layers': 1, 'Optimizer': 'adam', 'Epochs': 50}
Evaluating configuration: {'Activation Function': 'tanh', 'Number of Neurons': 32, 'Dropout Rate': 0.3, 'Number of Layers': 1, 'Optimizer': 'sgd', 'Epochs': 100}
Evaluating configuration

In [11]:
# Save results to CSV file
result_df = pd.DataFrame([
    {
        'Activation Function': res['Configuration']['Activation Function'],
        'Number of Neurons': res['Configuration']['Number of Neurons'],
        'Dropout Rate': res['Configuration']['Dropout Rate'],
        'Number of Layers': res['Configuration']['Number of Layers'],
        'Optimizer': res['Configuration']['Optimizer'],
        'Epochs': res['Configuration']['Epochs'],
        'Test MAE': res['Test MAE'],
        'Test Loss': res['Test Loss']
    }
    for res in results
])

# Save to a CSV file
result_df.to_csv('B180305023.csv', index=False)

# Output message when done
print("Results have been saved to 'B180305023.csv'")


Results have been saved to 'B180305023.csv'
